In [ ]:
import io, time, requests, pandas as pd
START, END = '2021-04-30', '2026-04-30'          # month ends to download
ASEAN = ['SGD','MYR','THB','PHP','IDR','VND','LAK','BND','MMK','KHR']
UA = {'User-Agent': 'Mozilla/5.0'}
URL = 'https://www.xe.com/currencytables/'       # source cited in the paper: XE Currency Exchange Rate Tables, base XDR (SDR)
def month_table(day):
    r = requests.get(URL, params={'from': 'XDR', 'date': day}, headers=UA, timeout=30)
    r.raise_for_status()
    t = next(t for t in pd.read_html(io.StringIO(r.text)) if 'Currency' in t.columns)
    col = next(c for c in t.columns if 'Units per' in str(c))          # units of currency per 1 SDR
    return t.set_index('Currency')[col].astype(float)

In [ ]:
rows = {}
for d in pd.date_range(START, END, freq='ME'):
    day = d.strftime('%Y-%m-%d')
    for attempt in range(3):
        try: rows[day] = month_table(day); break
        except Exception as e: print(day, 'retry', e); time.sleep(5)
    time.sleep(2)                                  # be polite to the server
w = pd.DataFrame(rows).T.rename_axis('date')
print(w.shape, 'months x currencies'); w.iloc[-3:, :6]

In [ ]:
with pd.ExcelWriter('QC_new.xlsx', engine='openpyxl') as xw:
    w.to_excel(xw, sheet_name='Data Dunia')
    w[['IDR']].to_excel(xw, sheet_name='Data IDR')
    w[ASEAN].to_excel(xw, sheet_name='Data ASEAN', startrow=1)
    ws = xw.sheets['Data ASEAN']; ws.cell(1, 1, 'date'); ws.cell(1, 2, 'Nilai Asli')
from google.colab import files; files.download('QC_new.xlsx')